# SMT End-to-End Pipeline

This notebook covers:
- Optional web crawling (for assignment evidence)
- Dataset loading (JSON/CSV/TSV/TXT)
- Text cleaning + tokenization
- Statistical Machine Translation training (IBM1-style)
- Saving training progress and checkpoints
- Resume training


In [1]:
import os, json, re, random, time, math, csv
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Set
from datetime import datetime

import pandas as pd

# ---------- Optional installs ----------
try:
    import jieba
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jieba"])
    import jieba

try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

# ==============================
# CONFIG
# ==============================
DATASET_PATH = r"C:\Users\Kevin\Desktop\Artificial Intelligent\Artificial-Intelligent\data\translation2019zh_train.json"
DATASET_TEXT_COLUMNS = ["chinese", "english"]
RUN_DIR = r"C:\Users\Kevin\Desktop\Artificial Intelligent\Artificial-Intelligent\model\smt_runs\zh_en_pbsmt"

# Data / training caps
MAX_SENTENCES       = 150000
MAX_SENT_LEN        = 20
SEED                = 42

# IBM1
IBM1_ITERS          = 10          # per direction
IGNORE_STOPWORDS    = False       # *** MUST be False for MT (learn 'the/is/to') ***
USE_IDF_WEIGHT      = True
ADD_NULL            = True        # include NULL token during alignment
DICE_TOPK_PER_TOKEN = 30
DICE_MIN_THRESH     = 0.01

# Phrase extraction
MAX_SRC_PHRASE_LEN  = 5           # was 4
PHRASE_TOPK_PER_SRC = 50          # was 20

# LM
LM_ORDER            = 3           # trigram
LM_ALPHA            = 0.4         # stupid backoff factor

# Decoder (with limited reordering)
BEAM_SIZE           = 8
W_PHRASE            = 1.0
W_LEX               = 0.5
W_LM                = 0.8
W_WORD_PENALTY      = -0.2
MAX_JUMP            = 3           # NEW: allow limited skips
DIST_PENALTY        = -0.3        # penalty per jump distance

# Eval
BLEU_SAMPLE_SIZE    = 800

# --- Progress tracking / checkpoint hygiene ---
RESET_CHECKPOINTS   = False  # set True to wipe existing IBM1 checkpoints before training
STATUS_PATH         = os.path.join(RUN_DIR, "status.json")  # human-readable training status

random.seed(SEED)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(os.path.join(RUN_DIR, "checkpoints"), exist_ok=True)

# --- OPTIONAL: wipe old checkpoints if requested ---
if RESET_CHECKPOINTS:
    ckdir = os.path.join(RUN_DIR, "checkpoints")
    for fn in os.listdir(ckdir):
        try:
            os.remove(os.path.join(ckdir, fn))
        except:
            pass
    try:
        os.remove(STATUS_PATH)
    except:
        pass

# ==============================
# Helpers
# ==============================
EN_STOP = set(nltk_stopwords.words("english"))

for w in ["你好", "谢谢", "世界", "中国", "我们", "学校", "喜欢", "学习", "英文", "中文","今天","天气"]:
    jieba.add_word(w)

def tok_zh_words(s: str) -> List[str]:
    return [t for t in jieba.lcut(str(s)) if t.strip()]

def tok_en_words(s: str) -> List[str]:
    toks = re.findall(r"\b\w+\b", str(s).lower())
    return [t for t in toks if (not IGNORE_STOPWORDS or t not in EN_STOP)]

def load_dataset(path: str, text_cols: List[str]) -> Tuple[List[str], List[str]]:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        try:
            df = pd.read_json(path, lines=True)
        except ValueError:
            df = pd.read_json(path)
    elif ext == ".csv":
        df = pd.read_csv(path)
    elif ext == ".tsv":
        df = pd.read_csv(path, sep="\t")
    else:
        raise ValueError(f"Unsupported dataset format: {ext}")
    if not all(c in df.columns for c in text_cols):
        raise ValueError(f"Columns {text_cols} not found. Available: {list(df.columns)}")
    df = df[text_cols].dropna().head(MAX_SENTENCES).reset_index(drop=True)
    return list(df[text_cols[0]]), list(df[text_cols[1]])

def bleu_corpus(refs: List[List[str]], hyps: List[List[str]], max_n=4) -> float:
    def ngrams(seq, n): return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]
    logs = []
    for n in range(1, max_n+1):
        match = total = 0
        for r, h in zip(refs, hyps):
            rc, hc = Counter(ngrams(r, n)), Counter(ngrams(h, n))
            total += sum(hc.values())
            for g, c in hc.items():
                match += min(c, rc.get(g, 0))
        logs.append(float("-inf") if total == 0 or match == 0 else math.log(match/total))
    ref_len = sum(len(r) for r in refs)
    hyp_len = sum(len(h) for h in hyps)
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len/max(hyp_len,1))
    gm = 0.0 if any(x == float("-inf") for x in logs) else math.exp(sum(logs)/len(logs))
    return bp * gm

def append_metrics(row: Dict[str, str]):
    path = os.path.join(RUN_DIR, "metrics.csv")
    write_header = not os.path.exists(path)
    with open(path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=sorted(row.keys()))
        if write_header: w.writeheader()
        w.writerow(row)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# --- status helpers ---
def _now_iso():
    return datetime.now().isoformat(timespec="seconds")

def update_status(stage: str, it: int, total_iters: int, last_ckpt: str, extra: dict = None):
    """Write a small JSON with current iteration, last checkpoint, and timestamp."""
    try:
        status = {}
        if os.path.exists(STATUS_PATH):
            with open(STATUS_PATH, "r", encoding="utf-8") as f:
                status = json.load(f)
        status[stage] = {
            "current_iter": it,
            "total_iters": total_iters,
            "last_checkpoint": last_ckpt,
            "updated_at": _now_iso()
        }
        if extra:
            status[stage].update(extra)
        with open(STATUS_PATH, "w", encoding="utf-8") as f:
            json.dump(status, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print("[status] WARN:", e)

def read_status():
    if not os.path.exists(STATUS_PATH):
        return {}
    with open(STATUS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

# ==============================
# Prepare data
# ==============================
src_raw, tgt_raw = load_dataset(DATASET_PATH, DATASET_TEXT_COLUMNS)
print("Loaded", len(src_raw), "pairs")

pairs = []
for z, e in zip(src_raw, tgt_raw):
    sw, tw = tok_zh_words(z), tok_en_words(e)
    if 0 < len(sw) <= MAX_SENT_LEN and 0 < len(tw) <= MAX_SENT_LEN:
        pairs.append((sw, tw))
random.shuffle(pairs)
print("After filters:", len(pairs))

# Build Dice candidate pruning (both directions share it per token)
src_df, tgt_df = Counter(), Counter()
pair_df = defaultdict(Counter)
for s_words, t_words in pairs:
    s_set, t_set = set(s_words), set(t_words)
    for s in s_set: src_df[s] += 1
    for t in t_set: tgt_df[t] += 1
    for s in s_set:
        for t in t_set:
            pair_df[s][t] += 1

candidates_s2t: Dict[str, Set[str]] = {}
for s, t_counts in pair_df.items():
    scored = []
    for t, freq in t_counts.items():
        dice = 2.0 * freq / (src_df[s] + tgt_df[t])
        if dice >= DICE_MIN_THRESH:
            scored.append((t, dice))
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates_s2t[s] = set(t for t, _ in scored[:DICE_TOPK_PER_TOKEN])

# reverse candidates
rev_pair_df = defaultdict(Counter)
for s, t_counts in pair_df.items():
    for t, c in t_counts.items():
        rev_pair_df[t][s] = c

candidates_t2s: Dict[str, Set[str]] = {}
for t, s_counts in rev_pair_df.items():
    scored = []
    for s, freq in s_counts.items():
        dice = 2.0 * freq / (tgt_df[t] + src_df[s])
        if dice >= DICE_MIN_THRESH:
            scored.append((s, dice))
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates_t2s[t] = set(s for s, _ in scored[:DICE_TOPK_PER_TOKEN])

# IDF for target side (for both directions)
def build_idf_on(tokens_list: List[List[str]]) -> Dict[str, float]:
    df_c = Counter()
    for toks in tokens_list:
        df_c.update(set(toks))
    N = len(tokens_list)
    idf = defaultdict(lambda: 1.0)
    for w, c in df_c.items():
        idf[w] = math.log(1 + (N / (1 + c))) + 1.0
    return idf

idf_en = build_idf_on([t for _, t in pairs])
idf_zh = build_idf_on([s for s, _ in pairs])

# ==============================
# IBM1 Training (with checkpoints)
# ==============================
def train_ibm1(pairs: List[Tuple[List[str], List[str]]],
               candidates: Dict[str, Set[str]],
               iters: int,
               dir_tag: str,
               idf_target: Dict[str, float],
               add_null=True) -> Dict[str, Dict[str, float]]:
    ckdir = os.path.join(RUN_DIR, "checkpoints")
    os.makedirs(ckdir, exist_ok=True)
    # Try resume
    tprobs = defaultdict(lambda: defaultdict(lambda: 1.0))
    start_it = 1
    resumes = sorted([n for n in os.listdir(ckdir) if n.startswith(f"ibm1_{dir_tag}_iter_")])
    if resumes:
        last = resumes[-1]
        tprobs = defaultdict(lambda: defaultdict(float))
        data = load_json(os.path.join(ckdir, last))
        for s, d in data.items():
            for t, p in d.items():
                tprobs[s][t] = float(p)
        start_it = int(re.findall(r"(\d+)", last)[-1]) + 1
        print(f"[IBM1 {dir_tag}] Resuming from {last}")
        update_status(f"ibm1_{dir_tag}", start_it-1, iters, last_ckpt=os.path.join(ckdir, last))

    NULL = "<NULL>"

    for it in range(start_it, iters+1):
        t0 = time.time()
        count = defaultdict(Counter)
        total = defaultdict(float)
        for s_words, t_words in pairs:
            t_set = set(t_words)
            if add_null: t_set = set(t_set) | {NULL}
            for s in s_words:
                cand_t = candidates.get(s, set())
                if add_null: cand_t = set(cand_t) | {NULL}
                t_valid = [t for t in t_set if t in cand_t] or ([NULL] if add_null else [])
                if not t_valid:
                    continue
                weights = {}
                denom = 0.0
                for t in t_valid:
                    w = tprobs[s][t]
                    if USE_IDF_WEIGHT:
                        w *= idf_target[t] if t != NULL else 1.0
                    weights[t] = w
                    denom += w
                if denom == 0.0:
                    eq = 1.0 / len(t_valid)
                    for t in t_valid: weights[t] = eq
                    denom = 1.0
                inv = 1.0 / denom
                for t, w in weights.items():
                    frac = w * inv
                    count[s][t] += frac
                    total[s] += frac

        for s in count:
            s_total = total[s] if total[s] > 0 else 1.0
            for t, c in count[s].items():
                tprobs[s][t] = c / s_total

        # quick intrinsic BLEU proxy: pick max-prob target per src token (no LM)
        sample = pairs if len(pairs) <= BLEU_SAMPLE_SIZE else random.sample(pairs, BLEU_SAMPLE_SIZE)
        hyps, refs = [], []
        for s_words, t_words in sample:
            hyp = []
            for s in s_words:
                cands = tprobs.get(s, {})
                if cands:
                    best = max([(tt, p) for tt, p in cands.items() if tt != NULL],
                               key=lambda kv: kv[1], default=(None,0))
                    if best[0]: hyp.append(best[0])
            hyps.append(hyp)
            refs.append(t_words)
        train_bleu = bleu_corpus(refs, hyps)
        append_metrics({"stage": f"ibm1_{dir_tag}", "iter": it, "bleu": f"{train_bleu:.6f}"})

        ckfile = os.path.join(ckdir, f"ibm1_{dir_tag}_iter_{it:03d}.json")
        save_json({s: dict(d) for s, d in tprobs.items()}, ckfile)
        update_status(f"ibm1_{dir_tag}", it, iters, last_ckpt=ckfile, extra={"bleu": round(train_bleu, 6)})
        print(f"[IBM1 {dir_tag}] Iter {it}/{iters} | BLEU={train_bleu*100:.2f} | {time.time()-t0:.1f}s")
    return tprobs

pairs_s2t = pairs
pairs_t2s = [(t, s) for s, t in pairs]

tprobs_s2t = train_ibm1(pairs_s2t, candidates_s2t, IBM1_ITERS, "s2t", idf_en, add_null=ADD_NULL)
tprobs_t2s = train_ibm1(pairs_t2s, candidates_t2s, IBM1_ITERS, "t2s", idf_zh, add_null=ADD_NULL)

save_json({s: dict(d) for s, d in tprobs_s2t.items()}, os.path.join(RUN_DIR, "ibm1_s2t_final.json"))
save_json({t: dict(d) for t, d in tprobs_t2s.items()}, os.path.join(RUN_DIR, "ibm1_t2s_final.json"))

print("\n=== Training status ===")
print(json.dumps(read_status(), ensure_ascii=False, indent=2))

# ==============================
# Viterbi Alignments + Symmetrization
# ==============================
NULL = "<NULL>"
NULL_PENALTY = 0.5  # penalize NULL so real links are preferred

def viterbi_align_one_s2t(s_words, t_words, tprobs):
    aligns = set()
    for i, s in enumerate(s_words):
        best_j, best_p = -1, 0.0
        for j, t in enumerate(t_words):
            p = tprobs.get(s, {}).get(t, 0.0)
            if p > best_p:
                best_p, best_j = p, j
        p_null = tprobs.get(s, {}).get(NULL, 0.0) * NULL_PENALTY
        if p_null >= best_p:
            continue
        if best_j >= 0:
            aligns.add((i, best_j))
    return aligns

def grow_diag_final_and(A, B, I, J):
    inter = A & B
    union = A | B
    S = set(inter)
    added = True
    neigh = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]
    while added:
        added = False
        for (i,j) in list(S):
            for di,dj in neigh:
                i2, j2 = i+di, j+dj
                if 0 <= i2 < I and 0 <= j2 < J:
                    if (i2,j2) in union and (i2,j2) not in S:
                        if (i2 not in [x for x,_ in S]) or (j2 not in [y for _,y in S]):
                            S.add((i2,j2)); added = True
    for (i,j) in union:
        if i not in [x for x,_ in S] or j not in [y for _,y in S]:
            S.add((i,j))
    return S

alignments_sym = []
ck_align_path = os.path.join(RUN_DIR, "checkpoints", "alignments_sym.jsonl")
with open(ck_align_path, "w", encoding="utf-8") as fout:
    for idx, (s_words, t_words) in enumerate(pairs):
        A = viterbi_align_one_s2t(s_words, t_words, tprobs_s2t)
        B_rev = viterbi_align_one_s2t(t_words, s_words, tprobs_t2s)
        B = {(j,i) for (i,j) in B_rev}
        S = grow_diag_final_and(A, B, len(s_words), len(t_words))
        alignments_sym.append(S)
        fout.write(json.dumps({"idx": idx, "s": s_words, "t": t_words, "a": sorted(list(S))}, ensure_ascii=False) + "\n")
print("Saved symmetrized alignments:", ck_align_path)

# ==============================
# Phrase Extraction (Koehn et al.)
# ==============================
def extract_phrases_for_sentence(s_words, t_words, align_set: Set[Tuple[int,int]], max_src_len=MAX_SRC_PHRASE_LEN):
    phrases = []
    I, J = len(s_words), len(t_words)
    aligned_to_t = defaultdict(set)
    aligned_to_s = defaultdict(set)
    for i,j in align_set:
        aligned_to_s[i].add(j)
        aligned_to_t[j].add(i)

    for i1 in range(I):
        for i2 in range(i1, min(I, i1 + max_src_len)):
            js = [j for i in range(i1, i2+1) for j in aligned_to_s.get(i, [])]
            if not js:
                continue
            j_min, j_max = min(js), max(js)
            # consistency
            out = False
            for j in range(j_min, j_max+1):
                for i in aligned_to_t.get(j, []):
                    if i < i1 or i > i2:
                        out = True; break
                if out: break
            if out: continue

            # expand over unaligned target words
            j1 = j_min
            while j1 >= 0 and (j1 not in aligned_to_t):
                j1 -= 1
            j1 += 1
            j2 = j_max
            while j2 < J and (j2 not in aligned_to_t):
                j2 += 1
            j2 -= 1
            for y1 in range(j1, j_min+1):
                for y2 in range(j_max, j2+1):
                    f = tuple(s_words[i1:i2+1])
                    e = tuple(t_words[y1:y2+1])
                    phrases.append((f, e))
    return phrases

phrase_counts = Counter()
src_phrase_total = Counter()

for (s_words, t_words), align in zip(pairs, alignments_sym):
    extracted = extract_phrases_for_sentence(s_words, t_words, align, max_src_len=MAX_SRC_PHRASE_LEN)
    for f, e in extracted:
        phrase_counts[(f, e)] += 1
        src_phrase_total[f] += 1

# φ(e|f)
phrase_table = defaultdict(lambda: defaultdict(float))
for (f, e), c in phrase_counts.items():
    phrase_table[f][e] = c / max(1, src_phrase_total[f])

# ==============================
# Lexical weights (using IBM1)
# ==============================
def lexical_weight_e_given_f(e: Tuple[str, ...], f: Tuple[str, ...], tprobs_s2t) -> float:
    prod = 1.0
    for ei in e:
        numer = sum(tprobs_s2t.get(fj, {}).get(ei, 0.0) for fj in f)
        denom = len(f)
        if numer == 0.0:
            numer = tprobs_s2t.get("<NULL>", {}).get(ei, 0.0)
            denom = 1
        prod *= max(numer / max(denom,1), 1e-12)
    return prod

lex_table = defaultdict(lambda: defaultdict(float))
for f, e_dict in phrase_table.items():
    for e in e_dict:
        lex_table[f][e] = lexical_weight_e_given_f(e, f, tprobs_s2t)

# Trim phrase table to top-K per source phrase (by φ * lex)
for f, e_dict in list(phrase_table.items()):
    scored = []
    for e, phi in e_dict.items():
        score = math.log(max(phi, 1e-12)) + 0.5 * math.log(max(lex_table[f][e], 1e-12))
        scored.append((e, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    keep = set([e for e, _ in scored[:PHRASE_TOPK_PER_SRC]])
    phrase_table[f] = {e: phrase_table[f][e] for e in keep}
    lex_table[f]   = {e: lex_table[f][e] for e in keep}

save_json({ "phi": { " ".join(f): { " ".join(e): v for e, v in ed.items() } for f, ed in phrase_table.items() },
            "lex": { " ".join(f): { " ".join(e): v for e, v in ed.items() } for f, ed in lex_table.items() } },
          os.path.join(RUN_DIR, "phrase_table.json"))
print("Saved phrase_table.json")

# ------------------------------
# Inject singleton back-off phrases from IBM1
# ------------------------------
BACKOFF_TOPK = 3

def ensure_singleton_backoff(tprobs_s2t, phrase_table, lex_table):
    for s in list(tprobs_s2t.keys()):
        if s == NULL:
            continue
        f = (s,)
        ranked = sorted([(t, p) for t, p in tprobs_s2t[s].items() if t != NULL],
                        key=lambda x: x[1], reverse=True)[:BACKOFF_TOPK]
        if not ranked:
            continue
        if f not in phrase_table:
            phrase_table[f] = {}
        if f not in lex_table:
            lex_table[f] = {}
        for t, p in ranked:
            e = (t,)
            phi = max(p, 1e-6)
            phrase_table[f][e] = max(phrase_table[f].get(e, 0.0), phi)
            lex_table[f][e] = max(lex_table[f].get(e, 0.0), max(p, 1e-12))

ensure_singleton_backoff(tprobs_s2t, phrase_table, lex_table)

# ==============================
# Simple trigram LM (stupid backoff)
# ==============================
BOS = "<s>"
EOS = "</s>"

def build_lm_trigram(corpus: List[List[str]]):
    unigrams = Counter(); bigrams = Counter(); trigrams = Counter()
    for toks in corpus:
        seq = [BOS, BOS] + toks + [EOS]
        for i in range(2, len(seq)):
            unigrams[seq[i]] += 1
            bigrams[(seq[i-1], seq[i])] += 1
            trigrams[(seq[i-2], seq[i-1], seq[i])] += 1
    total_unigrams = sum(unigrams.values())
    def logprob(nextw, w1, w2):
        tri = trigrams.get((w1, w2, nextw), 0)
        bi  = bigrams.get((w2, nextw), 0)
        uni = unigrams.get(nextw, 0)
        if tri > 0:
            denom = bigrams.get((w1, w2), 1)
            return math.log(tri / denom)
        elif bi > 0:
            denom = unigrams.get(w2, 1)
            return math.log(LM_ALPHA * bi / denom)
        else:
            return math.log(LM_ALPHA * LM_ALPHA * (uni + 1) / (total_unigrams + len(unigrams) + 1))
    return logprob, {"unigrams": unigrams, "bigrams": {" ".join(k): v for k,v in bigrams.items()},
                     "trigrams": {" ".join(k): v for k,v in trigrams.items()}}

lm_logprob, lm_counts = build_lm_trigram([t for _, t in pairs])
save_json(lm_counts, os.path.join(RUN_DIR, "lm_trigram_counts.json"))
print("Saved LM counts")

# ==============================
# Phrase-based decoder with limited jumps
# ==============================
# Build quick index: src phrase -> candidates with precomputed feature logs
src_phrase_index = {}
for f, e_dict in phrase_table.items():
    candidates = []
    for e, phi in e_dict.items():
        lp = math.log(max(phi, 1e-12))
        ll = math.log(max(lex_table[f][e], 1e-12))
        candidates.append((e, lp, ll))
    src_phrase_index[f] = candidates

from functools import lru_cache

def decode_with_jumps(s_words: List[str]) -> List[str]:
    N = len(s_words)
    # Precompute all phrase options by (start, length)
    span_options = defaultdict(list)  # (i, L) -> list[(e_tokens, lp, ll)]
    for i in range(N):
        for L in range(1, min(MAX_SRC_PHRASE_LEN, N - i) + 1):
            f = tuple(s_words[i:i+L])
            if f in src_phrase_index:
                span_options[(i, L)] = src_phrase_index[f]

    @lru_cache(maxsize=None)
    def search(mask: int, w1: str, w2: str):
        if mask == (1 << N) - 1:
            return ([], W_LM * lm_logprob(EOS, w1, w2))
        best_hyp, best_score = [], -1e9

        # leftmost uncovered
        pos = 0
        while pos < N and ((mask >> pos) & 1):
            pos += 1

        advanced = False
        # try covering at pos
        for L in range(1, min(MAX_SRC_PHRASE_LEN, N - pos) + 1):
            if ((mask >> pos) & 1) or any(((mask >> k) & 1) for k in range(pos, pos+L)):
                continue
            if (pos, L) not in span_options:
                continue
            advanced = True
            new_mask = mask | sum(1 << k for k in range(pos, pos+L))
            for e_tokens, lp, ll in span_options[(pos, L)]:
                lm_s = 0.0
                ww1, ww2 = w1, w2
                for tok in e_tokens:
                    lm_s += lm_logprob(tok, ww1, ww2)
                    ww1, ww2 = ww2, tok
                sub_hyp, sub_score = search(new_mask, ww1, ww2)
                score = sub_score + W_PHRASE*lp + W_LEX*ll + W_LM*lm_s + W_WORD_PENALTY*len(e_tokens)
                if score > best_score:
                    best_score = score
                    best_hyp = list(e_tokens) + sub_hyp

        # limited jump ahead
        for jump in range(1, MAX_JUMP + 1):
            jpos = pos + jump
            if jpos >= N: break
            if (mask >> jpos) & 1:
                continue
            for L in range(1, min(MAX_SRC_PHRASE_LEN, N - jpos) + 1):
                if any(((mask >> k) & 1) for k in range(jpos, jpos+L)):
                    continue
                if (jpos, L) not in span_options:
                    continue
                advanced = True
                new_mask = mask | sum(1 << k for k in range(jpos, jpos+L))
                for e_tokens, lp, ll in span_options[(jpos, L)]:
                    lm_s = 0.0
                    ww1, ww2 = w1, w2
                    for tok in e_tokens:
                        lm_s += lm_logprob(tok, ww1, ww2)
                        ww1, ww2 = ww2, tok
                    sub_hyp, sub_score = search(new_mask, ww1, ww2)
                    score = sub_score + W_PHRASE*lp + W_LEX*ll + W_LM*lm_s + W_WORD_PENALTY*len(e_tokens) + DIST_PENALTY*jump
                    if score > best_score:
                        best_score = score
                        best_hyp = list(e_tokens) + sub_hyp

        # back-off: IBM1 top-1 if nothing advanced
        if not advanced:
            s = s_words[pos]
            cands = tprobs_s2t.get(s, {})
            best = None
            for t, p in sorted(cands.items(), key=lambda kv: kv[1], reverse=True):
                if t != NULL:
                    best = (t, p); break
            if best:
                tok = best[0]
                lm_s = lm_logprob(tok, w1, w2)
                sub_hyp, sub_score = search(mask | (1 << pos), w2, tok)
                lp = math.log(max(best[1], 1e-12))
                ll = lp
                score = sub_score + W_PHRASE*lp + W_LEX*ll + W_LM*lm_s + W_WORD_PENALTY
                if score > best_score:
                    best_score = score
                    best_hyp = [tok] + sub_hyp

        return (best_hyp, best_score)

    hyp, _ = search(0, BOS, BOS)
    return hyp

# ==============================
# Evaluation (BLEU on a sample)
# ==============================
sample = pairs if len(pairs) <= BLEU_SAMPLE_SIZE else random.sample(pairs, BLEU_SAMPLE_SIZE)
hyps, refs = [], []
t0 = time.time()
for s_words, t_words in sample:
    hyp = decode_with_jumps(s_words)
    hyps.append(hyp)
    refs.append(t_words)
bleu = bleu_corpus(refs, hyps)
append_metrics({"stage": "pbsmt_decode_jump", "iter": 0, "bleu": f"{bleu:.6f}"})
print(f"[PBSMT+jump] BLEU={bleu*100:.2f} on {len(sample)} sents | {time.time()-t0:.1f}s")

# ==============================
# Analysis helpers
# ==============================
def topk_word_translations(src_tokens: List[str], tprobs: Dict[str, Dict[str, float]], k=5):
    print("\n=== Top-k word translation points (t(e|f)) ===")
    for s in src_tokens:
        cands = tprobs.get(s, {})
        ranked = sorted([(t, p) for t, p in cands.items() if t != NULL], key=lambda x: x[1], reverse=True)[:k]
        print(f"{s:>10} -> ", ", ".join([f"{t}:{p:.3f}" for t,p in ranked]) or "(none)")

def translate_and_explain(zh_sent: str):
    s_words = tok_zh_words(zh_sent)
    en = decode_with_jumps(s_words)
    print("\nZH:", zh_sent)
    print("EN:", " ".join(en))
    topk_word_translations(s_words, tprobs_s2t, k=5)

# Quick demo on a few sents
for demo in ["你好世界", "谢谢", "今天天气很好", "我们去学校", "我喜欢学习英文", "中国文化很有趣"]:
    translate_and_explain(demo)

print("\nArtifacts saved in:", RUN_DIR)
print(" - IBM1 checkpoints: RUN_DIR/checkpoints/ibm1_{s2t,t2s}_iter_XXX.json")
print(" - Symmetrized alignments: RUN_DIR/checkpoints/alignments_sym.jsonl")
print(" - Phrase table (with lexical weights): RUN_DIR/phrase_table.json")
print(" - Trigram LM counts: RUN_DIR/lm_trigram_counts.json")
print(" - Metrics CSV: RUN_DIR/metrics.csv")
print(" - Status file: RUN_DIR/status.json")


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Kevin\AppData\Local\Temp\jieba.cache
Loading model cost 0.597 seconds.
Prefix dict has been built successfully.


Loaded 150000 pairs
After filters: 67648
[IBM1 s2t] Resuming from ibm1_s2t_iter_010.json
[IBM1 t2s] Resuming from ibm1_t2s_iter_010.json

=== Training status ===
{
  "ibm1_s2t": {
    "current_iter": 10,
    "total_iters": 10,
    "last_checkpoint": "C:\\Users\\Kevin\\Desktop\\Artificial Intelligent\\Artificial-Intelligent\\model\\smt_runs\\zh_en_pbsmt\\checkpoints\\ibm1_s2t_iter_010.json",
    "updated_at": "2025-08-27T16:35:06"
  },
  "ibm1_t2s": {
    "current_iter": 10,
    "total_iters": 10,
    "last_checkpoint": "C:\\Users\\Kevin\\Desktop\\Artificial Intelligent\\Artificial-Intelligent\\model\\smt_runs\\zh_en_pbsmt\\checkpoints\\ibm1_t2s_iter_010.json",
    "updated_at": "2025-08-27T16:35:07"
  }
}
Saved symmetrized alignments: C:\Users\Kevin\Desktop\Artificial Intelligent\Artificial-Intelligent\model\smt_runs\zh_en_pbsmt\checkpoints\alignments_sym.jsonl
Saved phrase_table.json
Saved LM counts
[PBSMT+jump] BLEU=15.77 on 800 sents | 67.4s

ZH: 你好世界
EN: hello world

=== Top-k word

In [2]:
# ==============================
# Interactive Chinese → English prompt (and a simple function)
# ==============================

def zh2en(zh_text: str) -> str:
    """Translate a Chinese sentence to English using the trained PBSMT decoder."""
    s_words = tok_zh_words(zh_text)
    en = decode_with_jumps(s_words)
    return " ".join(en)

def translate_prompt(show_points: bool = True, k: int = 5):
    """
    Start a small REPL to translate Chinese to English.
    Commands:
      /q            quit
      /k N          set top-k for word translation points (default 5)
      /nop          hide word translation points
      /p            show word translation points
    """
    print("Chinese → English translator. Type Chinese and press Enter.")
    print("Commands: /q (quit), /k N (set top-k), /nop (hide points), /p (show points)\n")
    while True:
        try:
            zh = input("ZH> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            break

        if not zh:
            continue
        if zh in ("/q", "/quit", "/exit"):
            print("Bye.")
            break
        if zh.startswith("/k"):
            parts = zh.split()
            if len(parts) == 2 and parts[1].isdigit():
                k = max(1, int(parts[1]))
                print(f"Top-k now = {k}")
            else:
                print("Usage: /k N   (example: /k 10)")
            continue
        if zh == "/nop":
            show_points = False
            print("Will hide word translation points.")
            continue
        if zh == "/p":
            show_points = True
            print("Will show word translation points.")
            continue

        # translate
        s_words = tok_zh_words(zh)
        en_words = decode_with_jumps(s_words)
        print("EN>", " ".join(en_words))

        # optional per-word top-k translation points
        if show_points:
            print("\n=== Top-k word translation points (t(e|f)) ===")
            for s in s_words:
                cands = tprobs_s2t.get(s, {})
                ranked = sorted(
                    [(t, p) for t, p in cands.items() if t != "<NULL>"],
                    key=lambda x: x[1],
                    reverse=True
                )[:k]
                if ranked:
                    print(f"{s:>10} -> " + ", ".join([f"{t}:{p:.3f}" for t, p in ranked]))
                else:
                    print(f"{s:>10} -> (none)")
            print()

# --- quick examples ---
#print(zh2en("我们去学校"))
#print(zh2en("今天天气很好"))

# --- start interactive prompt ---
translate_prompt()


Chinese → English translator. Type Chinese and press Enter.
Commands: /q (quit), /k N (set top-k), /nop (hide points), /p (show points)

EN> we

=== Top-k word translation points (t(e|f)) ===
        我们 -> we:0.579, our:0.123, us:0.059, the:0.040, to:0.009

EN> the you

=== Top-k word translation points (t(e|f)) ===
         在 -> the:0.279, in:0.228, a:0.005, of:0.004, to:0.004
         吗 -> you:0.347, i:0.002, do:0.001, can:0.001, have:0.000

EN> today i i eat ice

=== Top-k word translation points (t(e|f)) ===
        今天 -> today:0.652, i:0.021, morning:0.003, day:0.001, evening:0.000
         我 -> i:0.681, my:0.133, me:0.087, the:0.002, to:0.002
         想 -> to:0.379, i:0.288, you:0.059, want:0.023, think:0.006
         吃 -> eat:0.232, eating:0.017, ate:0.003, food:0.000, lunch:0.000
       冰淇淋 -> ice:0.770, cream:0.159, trev:0.071, chocolate:0.000, serving:0.000

EN> i eat ice

=== Top-k word translation points (t(e|f)) ===
         我 -> i:0.681, my:0.133, me:0.087, the:0.002, to:

In [18]:
jieba.del_word("今天天气")